In [13]:
# =============================================================
# 0. Import libraries
# =============================================================
import pandas as pd
import numpy as np
import pymysql
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
from scipy.sparse import hstack
from collections import Counter
from imblearn.over_sampling import RandomOverSampler
import pickle


# =============================================================
# 1. Connect to RDS
# =============================================================
conn = pymysql.connect(
    host="amazonsql.cepao6y6sh9u.us-east-1.rds.amazonaws.com",
    user="admin",
    password="amazondb",
    port=3306,
    database="amazon"
)


# =============================================================
# 2. Load Tables
# =============================================================
ship = pd.read_sql("SELECT * FROM shipments;", conn)
label = pd.read_sql("SELECT * FROM fulfillment_error_label;", conn)
env = pd.read_sql("SELECT * FROM dim_environment;", conn)
op = pd.read_sql("SELECT * FROM dim_operational;", conn)
orders = pd.read_sql("SELECT * FROM orders;", conn)
cust = pd.read_sql("SELECT * FROM customers;", conn)


# =============================================================
# 3. Merge tables
# =============================================================
ship["ship_date"] = pd.to_datetime(ship["ship_date"])
env["date_key"] = pd.to_datetime(env["date_key"])

df = ship.merge(label, on=["shipment_id", "order_id"], how="left")

df = df.merge(
    env,
    left_on=["ship_date", "warehouse_id"],
    right_on=["date_key", "region"],
    how="left"
)

df = df.merge(
    op,
    on=["carrier", "warehouse_id"],
    how="left"
)

df = df.merge(
    orders,
    left_on="order_id",
    right_on="Order_ID",
    how="left"
)

df = df.merge(
    cust,
    on="customer_id",
    how="left"
)


# =============================================================
# 4. Target variable
# =============================================================
y = df["error_class"]


# =============================================================
# 5. Drop useless / explosive cardinality columns
# =============================================================
drop_cols = [
    "shipment_id", "order_id", "Order_ID", "customer_id", "Customer_ID",
    "Shipment_ID", "Shipping_address_ID", "Payment_ID",
    "tracking_number",
    "error_class", "error_class_desc",
    "reason_code",
    "generated_at",
    "signup_date",
    "Order_date",
    "created_at_x", "created_at_y",
    "updated_at_x", "updated_at_y",
    "Updated_at",
    "date_key", "region"
]

X = df.drop(columns=[c for c in drop_cols if c in df.columns])


# =============================================================
# 6. Fix datetime → timestamp numeric
# =============================================================
date_cols = ["estimated_delivery", "actual_delivery"]

for col in date_cols:
    if col in X.columns:
        X[col] = pd.to_datetime(X[col], errors="coerce")
        X[col] = X[col].astype("int64") // 1_000_000_000  # seconds


# =============================================================
# 7. Define categorical + numeric columns
# =============================================================
cat_cols = [
    "carrier",
    "warehouse_id",
    "delivery_type",
    "delivery_status",
    "Order_status",
    "state",
    "preferred_language",
    "account_status",
]

cat_cols = [c for c in cat_cols if c in X.columns]

num_cols = [c for c in X.columns if c not in cat_cols]


# Convert numeric-like columns
for col in num_cols:
    X[col] = pd.to_numeric(X[col], errors="coerce")

X[num_cols] = X[num_cols].fillna(0).astype("float32")


# =============================================================
# 8. Sparse One-Hot Encoding
# =============================================================
enc = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
X_cat = enc.fit_transform(X[cat_cols])
X_num = X[num_cols].values.astype("float32")
X_full = hstack([X_cat, X_num])


# =============================================================
# 9. Train/Test Split
# =============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y, test_size=0.2, random_state=42, stratify=y
)


# =============================================================
# 10. Class Weights (automatic)
# =============================================================
counts = Counter(y_train)
total = sum(counts.values())
num_classes = len(counts)

class_weights = {cls: (total / (num_classes * cnt)) for cls, cnt in counts.items()}
print("Class weights:", class_weights)

sample_weight = y_train.map(class_weights)


# =============================================================
# 11. Oversampling (Random Oversampling)
# =============================================================
ros = RandomOverSampler(random_state=42)
X_train_os, y_train_os = ros.fit_resample(X_train, y_train)

# Sample weights after oversampling: safe to ignore or apply proportionally
sample_weight_os = None


# =============================================================
# 12. Tuned XGBoost Model (stronger)
# =============================================================
model = XGBClassifier(
    objective="multi:softprob",
    num_class=4,
    eval_metric="mlogloss",
    learning_rate=0.1,
    max_depth=8,
    n_estimators=400,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=1,
    min_child_weight=3,
    tree_method="hist"
)

model.fit(X_train_os, y_train_os)


# =============================================================
# 13. Evaluation
# =============================================================
y_pred = model.predict(X_test)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nMacro-F1:", f1_score(y_test, y_pred, average="macro"))


# =============================================================
# 14. Save model + encoder + feature pipeline metadata
# =============================================================

# 14.1 Save trained model
pickle.dump(model, open("model_tuned.pkl", "wb"))

# 14.2 Save OneHotEncoder
pickle.dump(enc, open("encoder_tuned.pkl", "wb"))

# 14.3 Save categorical columns
pickle.dump(cat_cols, open("cat_cols.pkl", "wb"))

# 14.4 Save numeric columns
pickle.dump(num_cols, open("num_cols.pkl", "wb"))

# 14.5 Save final feature ordering AFTER OHE
feature_names = list(enc.get_feature_names_out(cat_cols)) + num_cols
pickle.dump(feature_names, open("feature_names.pkl", "wb"))

print("\nSaved:")
print(" model_tuned.pkl")
print(" encoder_tuned.pkl")
print(" cat_cols.pkl")
print(" num_cols.pkl")
print(" feature_names.pkl")


C:\Users\minse\AppData\Local\Temp\ipykernel_17932\2370111296.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  ship = pd.read_sql("SELECT * FROM shipments;", conn)
C:\Users\minse\AppData\Local\Temp\ipykernel_17932\2370111296.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  label = pd.read_sql("SELECT * FROM fulfillment_error_label;", conn)
C:\Users\minse\AppData\Local\Temp\ipykernel_17932\2370111296.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  env = pd.read_sql("SELECT * FROM dim_environment;", conn)
C:\Users

Class weights: {0: 0.9642876084220879, 2: 0.7306416494965879, 3: 1.7556950357722862, 1: 0.9758668136972666}

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    103703
           1       0.40      0.84      0.54    102473
           2       0.83      0.26      0.39    136866
           3       0.20      0.13      0.16     56958

    accuracy                           0.58    400000
   macro avg       0.61      0.56      0.52    400000
weighted avg       0.67      0.58      0.56    400000


Confusion Matrix:
[[103703      0      0      0]
 [     0  85893   1181  15399]
 [     0  85923  35344  15599]
 [     0  43079   6294   7585]]

Macro-F1: 0.5233657164340477

Saved:
 model_tuned.pkl
 encoder_tuned.pkl
 cat_cols.pkl
 num_cols.pkl
 feature_names.pkl
